In [ ]:
import os
import time
from dotenv import load_dotenv
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Property, DataType
from weaviate.classes.query import Filter
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from langchain_weaviate.vectorstores import WeaviateVectorStore
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_core.retrievers import BaseRetriever
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
from typing import List, Literal
import asyncio

from qdrant_client.models import (
    Distance,
    VectorParams
)

from langchain_groq import ChatGroq

from langchain.chains import (
    create_history_aware_retriever,
    create_retrieval_chain
)


from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)


from langchain_community.chat_message_histories import (
    RedisChatMessageHistory
)
from langgraph.prebuilt import create_react_agent

from langchain_core.tools import tool

In [ ]:
# API Keys Setup

load_dotenv()

weaviate_url = os.getenv("weaviate_url")
weaviate_api_key = os.getenv("weaviate_api_key")
qdrant_url= os.getenv("qdrant_url")
qdrant_api_key= os.getenv("qdrant_api_key")
groq_api_key= os.getenv("groq_api_key") 
redis_url = os.getenv("redis_url")

In [ ]:
# Connecting to Vector DB for Sementic Memory

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate.auth.AuthApiKey(weaviate_api_key),
    skip_init_checks=True
)

print(client.is_ready())  

In [ ]:
qdrant_client = QdrantClient(
    url = qdrant_url,
    api_key = qdrant_api_key
)
print(qdrant_client.get_collections())

In [ ]:
# Config for embedding model

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2" # for sementic memory
)

embedding_model_2 = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5" # for External RAG
)

In [ ]:
# Reranker and its Configuration

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

def rerank_documents(
    query: str,
    docs: list,
    top_k: int = 5
):

    if not docs:
        return []

    pairs = [
        [query, doc.page_content]
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    ranked_docs = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc
        for _, doc in ranked_docs[:top_k]
    ]

In [ ]:
# Setting up Vector Store for sementic memory

vectorStore = WeaviateVectorStore(
    client=client,
    index_name="Memory",
    text_key="content",
    embedding=embedding_model
)

In [ ]:
# Setting up Vector Store for External RAG

knowledge_vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name="KnowledgeBase",
    embedding=embedding_model_2,
    content_payload_key="text"
)

In [ ]:
# MAIN MODEL
llm = ChatGroq(
    model="llama-3.1-70b-versatile",
    api_key=groq_api_key,
    temperature=0.7
)

# MEMORY RETRIEVER
memory_retriever = vectorStore.as_retriever(
    search_kwargs={
        "k": 3
    }
)

# KNOWLEDGE RETRIEVER
knowledge_retriever = knowledge_vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 5,
        "score_threshold": 0.2
    }
)

class RewrittenQuery(BaseModel):
    query: str

context_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
Given a chat history and the latest user question,
rewrite the question so it can be understood independently.
Return only the rewritten question.
"""
    ),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

query_rewriter = (
    context_prompt
    | llm.with_structured_output(
        RewrittenQuery
    )
)

In [ ]:
# Tools for agent
from langchain.agents import create_tool_calling_agent
@tool
async def search_memory(
    query: str
) -> str:
    """
    Search user semantic memory such as preferences,
    interests, past conversations and personal facts.
    """

    docs = await vectorStore.asimilarity_search(
        query=query,
        k=3,
        filters=Filter.by_property(
            "user_id"
        ).equal("rohan_123")
    )

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )
    
@tool
async def search_knowledge(query: str) -> str:
    """
    Search technical knowledge base containing AI,
    machine learning, system design and engineering documents.
    """

    docs = await knowledge_retriever.ainvoke(
        query
    )

    docs = rerank_documents(
        query,
        docs,
        top_k=3
    )

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )

@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.
    """
    return str(eval(expression))



agent = create_react_agent(
    llm,
    [
        search_memory,
        search_knowledge,
        calculator
    ]
)

In [ ]:
# Semantic memory Extractor

memory_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key = groq_api_key,
    temperature=0
)

class Memory(BaseModel):

    content: str = Field(
        description="Important semantic memory extracted from conversation"
    )

    memory_type: Literal[
        "preference",
        "skill",
        "interest",
        "goal",
        "project"
    ] = Field(
        description="Type of memory"
    )

class MemoryExtraction(BaseModel):

    should_store: bool = Field(
        description="Whether conversation contains important memory"
    )

    memories: List[Memory] = Field(
        description="List of extracted semantic memories"
    )

structured_memory_llm = memory_llm.with_structured_output(
    MemoryExtraction
)

memory_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a semantic memory extraction system.

Extract ONLY important long-term user information.

Store:
- preferences
- skills
- interests
- goals
- ongoing projects

Ignore:
- greetings
- temporary discussion
- casual conversation

IMPORTANT RULES:
- Memories must be fully self-contained sentences.
- Memories must always start with "User".
- Memories must be semantically meaningful and retrieval-friendly.
- Do not store vague keywords.
- Rewrite extracted memories into natural semantic statements.

Good examples:
- "User has interest in deep learning"
- "User prefers C++ for coding"
- "User is learning distributed AI systems"

Bad examples:
- "deep learning"
- "C++"
- "distributed systems"
"""
    ),

    ("human", "{input}")
])

memory_chain = memory_prompt | structured_memory_llm

def get_session_history(session_id: str):
    return RedisChatMessageHistory(
        session_id=session_id,
        url = redis_url
    )

In [ ]:
# For Testing Purpose

# response = memory_chain.invoke({
#    "input": "I love deep learning and prefer coding in C++."
#})
# print(response.model_dump_json())

In [ ]:
# Checks whether related sementic memory exist in vector DB 

def memory_exists(memory_text, user_id, threshold=0.90):
    results = vectorStore.similarity_search_with_score(
        query=memory_text,
        k=1,
        filters=Filter.by_property("user_id").equal(user_id)
    )
    if not results:
        return False
    document, score = results[0]

    # print("MATCH:", document.page_content)
    # print("SCORE:", score)

    return score < threshold

In [ ]:
#Storing semantic user memory to Vector DB

def store_memories(memories, user_id):
    texts = []
    metadatas = []
    for memory in memories:
        if memory_exists(
            memory.content,
            user_id
        ):

            # print(f"Skipping duplicate: {memory.content}")

            continue
        texts.append(memory.content)
        metadatas.append({
            "memory_type": memory.memory_type,
            "user_id": user_id
        })
    if texts:

        vectorStore.add_texts(
            texts=texts,
            metadatas=metadatas
        )

In [ ]:
async def chat(
    user_input,
    session_id
):

    history = get_session_history(
        session_id
    )

    rewritten = await query_rewriter.ainvoke(
        {
            "input": user_input,
            "chat_history": history.messages
        }
    )

    response = await agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": rewritten.query
                }
            ]
        }
    )

    return response
    
await chat(user_input = "Can you explain me XAi",session_id="session_1")

In [ ]:
llm.invoke("hello")

In [ ]:
llm_with_tools = llm.bind_tools(
    [search_knowledge]
)

await llm_with_tools.ainvoke(
    "Explain XAI"
)

In [ ]:
await search_knowledge.ainvoke(
    {
        "query": "Explain XAI"
    }
)

In [ ]:
llm_with_tools = llm.bind_tools(
    [search_knowledge]
)

response = await llm_with_tools.ainvoke(
    "Explain xAI"
)

print(response)